In [ ]:
%pip install -q langchain==0.2.16 langchain-core==0.2.38 langchain-community==0.2.16 langchain-openai==0.1.23 langchain-text-splitters==0.2.4 langchain-groq==0.1.9 sentence-transformers chromadb gradio python-dotenv tiktoken
print("Done!")

In [ ]:
# We start by installing the libraries we need and setting up our OpenAI API key
print("Installing necessary libraries...")
%pip install -q sentence-transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("HuggingFace Embeddings initialized.")
print("Libraries installed successfully!")

In [ ]:
import os
from groq import Groq

groq_api_key = "your_groq_api_key_here"  # paste your key directly

client = Groq(api_key=groq_api_key)
print("Groq client successfully configured.")

In [ ]:
# Let's import Langchain components
from langchain_openai import OpenAIEmbeddings, OpenAI
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.chains import RetrievalQAWithSourcesChain

In [ ]:
# Define the path to your data file
# Ensure 'medical_qa_data.txt' is in the same folder as this notebook
DATA_FILE_PATH = "medical_qa_data.txt"
print(f"Data file path set to: {DATA_FILE_PATH}")

In [ ]:
# Let's load Eleven Madison Park Restaurant data, which has been scraped from their website
# The data is saved in "medical_qa_data.txt", Langchain's TextLoader makes this easy to read
print(f"Attempting to load data from: {DATA_FILE_PATH}")

# Initialize the TextLoader with the file path and specify UTF-8 encoding
# Encoding helps handle various characters correctly
loader = TextLoader(DATA_FILE_PATH, encoding = "utf-8")

# Load the document(s) using TextLoader from LangChain, which loads the entire file as one Document object
raw_documents = loader.load()
print(f"Successfully loaded {len(raw_documents)} document(s).")


In [ ]:
# Let's display a few characters of the loaded content to perform a sanity check!
print(raw_documents[0].page_content[:500] + "...")

In [ ]:
# Let's split the document into chunks
print("\nSplitting the loaded document into smaller chunks...")

# Let's initialize the splitter, which tries to split the document on common separators like paragraphs (\n\n),
# sentences (.), and spaces (' ').
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000,  # Aim for chunks of about 1000 characters
                                               chunk_overlap = 150,)  # Each chunk overlaps with the previous by 150 characters

# Split the raw document(s) into smaller Document objects (chunks)
documents = text_splitter.split_documents(raw_documents)

# Check if splitting produced any documents
if not documents:
    raise ValueError("Error: Splitting resulted in zero documents. Check the input file and splitter settings.")
print(f"Document split into {len(documents)} chunks.")


In [ ]:
documents

In [ ]:
# Let's display an example chunk and its metadata
print("\n--- Example Chunk (Chunk 2) ---")
print(documents[2].page_content)
print("\n--- Metadata for Chunk 2 ---")
print(documents[2].metadata) # Should show {'source': 'eleven_madison_park_data.txt'}

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("HuggingFace Embeddings initialized.")

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama3-8b-8192",
    api_key=groq_api_key,
    temperature=0
)
print("Groq LLM initialized.")

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("Initializing HuggingFace Embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("HuggingFace Embeddings initialized.")

print("\nCreating ChromaDB vector store and embedding documents...")
vector_store = Chroma.from_documents(documents=documents, embedding=embeddings)

vector_count = vector_store._collection.count()
print(f"ChromaDB vector store created with {vector_count} items.")

if vector_count == 0:
    raise ValueError("Vector store creation resulted in 0 items. Check previous steps.")

In [ ]:
# Let's retrieve the first chunk of stored data from the vector store
stored_data = vector_store._collection.get(include=["embeddings", "documents"], limit = 1)  

# Display the results
print("First chunk text:\n", stored_data['documents'][0])
print("\nEmbedding vector:\n", stored_data['embeddings'][0])
print(f"\nFull embedding has {len(stored_data['embeddings'][0])} dimensions.")

In [ ]:
# Let's perform a similarity search in our vector store
print("\n--- Testing Similarity Search in Vector Store ---")
test_query = "What are the symptoms of a heart attack?"
print(f"Searching for documents similar to: '{test_query}'")


# Perform a similarity search. 'k=2' retrieves the top 2 most similar chunks
try:
    similar_docs = vector_store.similarity_search(test_query, k = 2)
    print(f"\nFound {len(similar_docs)} similar documents:")

    # Display snippets of the retrieved documents and their sources
    for i, doc in enumerate(similar_docs):
        print(f"\n--- Document {i+1} ---")
        # Displaying the first 700 chars for brevity
        content_snippet = doc.page_content[:700].strip() + "..."
        source = doc.metadata.get("source", "Unknown Source")  # Get source from metadata
        print(f"Content Snippet: {content_snippet}")
        print(f"Source: {source}")

except Exception as e:
    print(f"An error occurred during similarity search: {e}")



In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQAWithSourcesChain

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # updated model name
    api_key="paste_your_groq_api_key_here", 
    temperature=0
)

qa_chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    verbose=True
)
print("RAG chain created successfully!")

In [ ]:
%pip install -q gradio==3.50.2

In [ ]:
from groq import Groq
client = Groq(api_key="paste_your_groq_api_key_here")
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "say hello"}]
)
print(response.choices[0].message.content)

In [ ]:
def ask_medical_assistant(question):
    result = qa_chain.invoke({"question": question})
    answer = result.get("answer", "No answer found.")
    sources = result.get("sources", "No sources.")
    print(f"Question: {question}")
    print(f"\nAnswer: {answer}")
    print(f"\nSources: {sources}")
    print("-" * 60)

# Test questions
ask_medical_assistant("What are the symptoms of a heart attack?")
ask_medical_assistant("How do I manage type 2 diabetes?")
ask_medical_assistant("What causes high blood pressure?")